In [71]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_openai import ChatOpenAI

from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage
from langgraph.graph.message import add_messages

import requests

from dotenv import load_dotenv
import os

In [60]:
load_dotenv()

True

In [61]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    

In [62]:
model = ChatOpenAI(
    model = "openai/gpt-oss-120b:free",
    openai_api_key = os.environ["OPENAI_API_KEY"],
    openai_api_base = os.environ["OPENAI_API_BASE"]
)

In [63]:
pip install -U ddgs

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [74]:
search_tool = DuckDuckGoSearchRun(reqion="us-en")

@tool
def calc(num1: float, num2: float, op: str):
    """ Used for doing the arthematic operation on two numbers
        parameters:
            num1 : first number
            num2 : second number
            op : operations performed
    """
    try:
        if(op == "add"):
            result = num1+num2
        elif(op == "subract"):
            result = num1-num2
        elif(op == "divide"):
            if(num2 == 0):
                return{"error": "Division by zero not defined"}
            result = num1/num2
        elif(op == "multiply"):
            result = num1*num2
        else:
            return {"error": "Invalid choice"}
        
        return {"num1": num1, "num2": num2, "operation": op, "result": result}
    except Exception as e:
        return {"error": str(e)}
    
@tool
def get_stock_price(symbol: str)->dict:
    """This Function is built for finding the stock price of a given company
    """

    url = (
        "https://www.alphavantage.co//query?",
        "function=GLOBAL_QUOTE&",
        f"symbol={symbol}",
        f"apikey={os.environ["ALPHA_VANTAGE_API"]}"
    )

    r = requests.get(url)
    return r.json()

In [75]:
#Creating the LLM with tools
tools = [search_tool, get_stock_price, calc]
llm_with_tools = model.bind_tools(tools)

In [76]:
def get_chat(state: ChatState):
    query = state["messages"]
    response = llm_with_tools.invoke(query)
    content = response.content
    return {"messages": content}

t_node = ToolNode(tools)

In [77]:
graph = StateGraph(ChatState)

graph.add_node("chat_node", get_chat)
graph.add_node("tools", t_node)

graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", tools_condition)

workflow = graph.compile()

In [80]:
initial_state = {"messages": [HumanMessage(content="what is the name of the current prime minister of india")]}
for step in workflow.stream(initial_state):
    print("STEP-", step)
    print("---")

STEP- {'chat_node': {'messages': ''}}
---


In [ ]:
initial_state = {"messages": [HumanMessage(content="what is the name of the current prime minister of india")]}
final_state = workflow.invoke(initial_state)
print(final_state["messages"][-1].content)